# Fine motor study: review and exploratory analysis

This notebook reviews acquired trials for **control vs disease**, under `normal_dom`, `normal_ndom`, `blind_dom`, and `blind_ndom`. Run from top to bottom, then edit the configuration and rerun as needed. `main.ipynb` acquires data; the Python modules beside it implement analysis and replay.

**Task:** start with hands at rest at shoulder width, pick up a lock and key, unlock it, then return to rest.

**What the data can tell us:** wrist-relative fingertip movement, thumb–index spacing, movement variability, and acquisition quality. Without a specified target/task, dispersion is **not target error or a validated precision score**. Intentional movement, hand rotation, camera noise, and tracking error all contribute. These outputs are exploratory study summaries, not a diagnostic classifier.

The saved x/y positions are normalized image coordinates; z is estimated depth relative to the wrist, not a calibrated physical point cloud. No millimetres or absolute 3D speed are inferred. See [MediaPipe coordinates](https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker/python).


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from proj.session import DEFAULT_DATA_DIR
from proj.review_data import discover_trials, prepare_group_file
from proj.motor_metrics import extract_metrics, PRIMARY_METRICS, METRICS
from proj.study_statistics import (
    subject_summary, descriptive_statistics, compare_groups, paired_condition_effects, balanced_overall_summary,
)
from proj.review_plots import plot_group_distributions, plot_paired_conditions, display_figure_once
from proj.landmark_replay import review_widget
from proj.task_analysis import add_task_metrics, TASK_OUTCOMES, outcome_counts

DATA_DIR = DEFAULT_DATA_DIR
EXPORT_DIR = DATA_DIR.parent / "analysis_exports"

# Optional overrides; dominant hand is normally loaded from recording metadata/subject CSV.
# Example: {"Alice": "right", "Bob": "left"}
DOMINANT_HANDS = {}
# Override a specific trial if needed: {"Alice/trial_folder": "right"}
HAND_OVERRIDES = {}
# Legacy CSV hand indices are not stable identities. Enable only after visual review.
ALLOW_LEGACY_HAND_INDEX = False
LEGACY_IMAGE_SIZE = None  # e.g. (640, 480), only if these were the actual capture dimensions

# Pre-specify quality thresholds and outcomes before inspecting group differences.
MIN_FRAMES = 30
MIN_COVERAGE = 0.8
MIN_DURATION_SECONDS = 1.0
MIN_HANDEDNESS_CONFIDENCE = 0.6
MAX_DERIVATIVE_GAP_SECONDS = 0.25
OUTCOMES = list(TASK_OUTCOMES)
WHOLE_TRIAL_OUTCOMES = ["recording_duration_seconds", *METRICS]
SUCCESSFUL_TASKS_ONLY = True  # Success is assumed for the single-event protocol.
MIN_REST_PHASE_FRAMES = 10


## 1. Recordings and group assignment

New recordings prompt once for `control` or `disease` and also ask which hand is dominant, and save both in metadata and `data/subject_groups.csv`. The CSV has `subject_name`, `group`, and `dominant_hand` columns; use `left` or `right` for handedness. For old recordings, this cell creates a subject/group template. Edit its blank group cells, save, and rerun this cell. The registry takes precedence over trial metadata, so any corrected assignment applies consistently to all that subject's trials. Unassigned subjects are visible for review but excluded from group inference. Subject names are matched exactly.

Cancelled/interrupted recordings remain in the inventory but are excluded from analysis. Existing directories in both naming schemes are discovered.


In [ ]:
inventory = discover_trials(DATA_DIR)
group_file = prepare_group_file(inventory, DATA_DIR)
inventory = discover_trials(DATA_DIR)
print(f"Group registry: {group_file}")
print(f"Found {len(inventory)} trials, {inventory.subject_name.nunique()} subjects.")
display(inventory)
if inventory.empty:
    print("No acquired trials yet. Record with main.ipynb, then rerun this notebook.")
else:
    display(inventory.groupby(["group", "experiment_type", "status"], dropna=False).size().rename("trials").reset_index())
    missing_groups = inventory.loc[inventory.group == "", "subject_name"].unique()
    if len(missing_groups):
        print(f"Assign groups in the registry: {list(missing_groups)}")


## 2. Quality and movement features

Choose the **task hand**, not detection index 0/1. Dominant hand comes from `subject_groups.csv` or recording metadata; `DOMINANT_HANDS` can override it. The `_dom`/`_ndom` condition selects the corresponding hand. Otherwise, auto-selection is allowed only when exactly one anatomical label is present. Ambiguous labels and low-confidence detections are excluded. Legacy indices require explicit opt-in because hand identity may switch.

Measurements use the 2D index fingertip relative to the wrist, correcting y for image aspect ratio. All distances use the trial's median wrist-to-middle-MCP length as a fixed scale:

- `tip_dispersion_rms`: RMS distance of the relative tip from its trial centroid (palm lengths). Appropriate as a variability measure; it mixes intentional movement with instability.
- `pinch_sd`: sample SD of thumb–index spacing (palm lengths). Also affected by the task's intentional opening/closing.
- `pinch_mean`: mean thumb–index spacing.
- `tip_speed_mean`: time-weighted relative fingertip speed (palm lengths/s).
- `tip_path_length`: summed relative motion over consecutive valid frames (palm lengths); strongly depends on trial duration.

Derivatives never bridge missing frames or long time gaps. No smoothing or tremor-frequency claim is made. Coverage measures selected-hand detections / recorded frames. New recordings save image dimensions and handedness; legacy recordings need known camera dimensions before computing metrics. Whole-trial motion features remain exploratory descriptors. Event outcomes are time from recording start to key insertion, time after insertion to recording end, and whole-trial duration. These include operator reaction/marking time and are not isolated unlocking duration. Thresholds are configurable and are not clinically validated cutoffs.

### Single event: key is in the lock

Space starts the countdown, then records **key is in the lock**, then finishes the trial and loads the next one. Only insertion is logged as an event. Success is assumed; there is no success/failure prompt. The time is saved in `task_annotations.json` and metadata as `key_in_lock_seconds`.

The event comes from the operator's key press, not landmark-based inference. For older trials or corrections, enter its time in the replay controls. Earlier four-phase recordings remain readable, but their phase markers are not reinterpreted as key insertion. Missing insertion times remain unavailable.


In [ ]:
hand_selections = dict(HAND_OVERRIDES)
for _, trial in inventory.iterrows():
    dominant = DOMINANT_HANDS.get(trial.subject_name) or trial.dominant_hand
    if dominant:
        if dominant not in ("left", "right"):
            raise ValueError("DOMINANT_HANDS values must be left or right")
        selected = ("left" if dominant == "right" else "right") if trial.experiment_type.endswith("_ndom") else dominant
        hand_selections.setdefault(trial.trial_id, selected)

trial_metrics = extract_metrics(
    inventory, hand_selections=hand_selections,
    allow_legacy=ALLOW_LEGACY_HAND_INDEX, legacy_image_size=LEGACY_IMAGE_SIZE,
    min_frames=MIN_FRAMES, min_coverage=MIN_COVERAGE,
    min_duration_seconds=MIN_DURATION_SECONDS,
    min_confidence=MIN_HANDEDNESS_CONFIDENCE,
    max_gap_seconds=MAX_DERIVATIVE_GAP_SECONDS,
)
trial_metrics = add_task_metrics(
    trial_metrics, hand_selections=hand_selections,
    allow_legacy=ALLOW_LEGACY_HAND_INDEX, legacy_image_size=LEGACY_IMAGE_SIZE,
    min_confidence=MIN_HANDEDNESS_CONFIDENCE,
    max_gap_seconds=MAX_DERIVATIVE_GAP_SECONDS, min_phase_frames=MIN_REST_PHASE_FRAMES,
)
display(trial_metrics)
display(trial_metrics[["trial_id", "annotation_status", "task_outcome", "phase_quality_note", *TASK_OUTCOMES]])
display(outcome_counts(trial_metrics))
display(trial_metrics.loc[~trial_metrics.analysis_eligible,
    ["trial_id", "status", "selected_hand", "detection_coverage", "exclusion_reason"]])
print(f"Eligible trials: {int(trial_metrics.analysis_eligible.sum())} / {len(trial_metrics)}")


## Whole-trial statistics and overall group summary

The per-trial table summarizes the entire CSV recording: total recording duration, selected-hand frames, coverage, fingertip dispersion, pinch mean/SD, relative speed, and path length. Whole-trial duration includes initial/final rest and operator waiting; active task duration is reported separately above. Whole-trial motion is an exploratory descriptor, not lock alignment error.

Control/disease descriptive statistics and comparisons below use one averaged trial value per subject and condition, with the same quality checks. They do **not** require phase annotations or a known success label. Cancelled/interrupted trials stay visible in the trial table but are excluded from inference.

The overall summary gives equal weight to the four condition means for each subject. Only subjects with all four eligible conditions contribute, preventing unequal condition mixtures between groups. A metric remains unavailable unless all four condition values exist. Holm correction is applied separately to the whole-trial condition comparisons and the overall comparisons.


In [ ]:
whole_trial_table = trial_metrics[[
    "trial_id", "subject_name", "group", "experiment_type", "status",
    "recording_duration_seconds", "frames_recorded", "selected_frames",
    "detection_coverage", *METRICS, "analysis_eligible", "exclusion_reason",
]]
display(whole_trial_table)
whole_trial_subjects = subject_summary(trial_metrics, metrics=WHOLE_TRIAL_OUTCOMES)
whole_trial_descriptives = descriptive_statistics(whole_trial_subjects, metrics=WHOLE_TRIAL_OUTCOMES)
whole_trial_comparisons = compare_groups(whole_trial_subjects, metrics=WHOLE_TRIAL_OUTCOMES)
display(whole_trial_descriptives)
display(whole_trial_comparisons)

overall_subjects = balanced_overall_summary(whole_trial_subjects, metrics=WHOLE_TRIAL_OUTCOMES)
overall_descriptives = descriptive_statistics(overall_subjects, metrics=WHOLE_TRIAL_OUTCOMES)
overall_comparisons = compare_groups(overall_subjects, metrics=WHOLE_TRIAL_OUTCOMES, conditions=["all_conditions"])
display(overall_subjects)
display(overall_descriptives)
display(overall_comparisons)
if overall_subjects.empty:
    print("Overall comparisons need subjects with all four eligible conditions.")


## 3. Subject-level descriptive statistics and figures

Each subject contributes one averaged value per condition, regardless of repeated trials or the number of frames. Report group subject counts, condition completeness, and exclusions alongside results. The rows below show mean, SD, median, and quartiles; dots in the figures are subjects, not frames.

New recordings assume success. Event statistics require an insertion marker; whole-trial summaries above do not. Each participant contributes one value per condition, regardless of trial or frame count. Time-to-key includes initial waiting after the countdown, and post-key time includes unlocking, returning to rest, and any final waiting. Neither is claimed to be a separately timed physical phase.


In [ ]:
analysis_trials = trial_metrics.copy()
# Manual timing can be usable even when hand landmarks fail QC.
for metric in OUTCOMES:
    if metric not in TASK_OUTCOMES:
        analysis_trials.loc[~trial_metrics.analysis_eligible, metric] = float("nan")
valid_timing = (analysis_trials.status == "completed") & (analysis_trials.annotation_status == "annotated") & (analysis_trials.error == "")
analysis_trials["analysis_eligible"] = analysis_trials.analysis_eligible | valid_timing
if SUCCESSFUL_TASKS_ONLY:
    analysis_trials.loc[analysis_trials.task_outcome != "success", "analysis_eligible"] = False
subjects = subject_summary(analysis_trials, metrics=OUTCOMES)
descriptives = descriptive_statistics(subjects, metrics=OUTCOMES)
display(subjects)
display(descriptives)
if not subjects.empty:
    display(subjects.groupby("group").subject_name.nunique().rename("unique_subjects"))
    display(subjects.pivot(index=["subject_name", "group"], columns="experiment_type", values="trial_count").fillna(0))
    distribution_figure = plot_group_distributions(subjects, metrics=OUTCOMES)
    display_figure_once(distribution_figure)
    plt.close("all")
    paired_figure = plot_paired_conditions(subjects, metric=OUTCOMES[0])
    display_figure_once(paired_figure)
    plt.close("all")
else:
    print("Group summaries need eligible, annotated trials with control/disease labels. Use the replay section to annotate, then rerun from metrics.")


## 4. Control vs disease, separately by condition

Welch's two-sided t-test compares independent **subject** means and allows unequal group variances. The effect direction is **disease minus control**. Results include a 95% CI for the mean difference and Hedges' g (approximate small-sample correction). Mann–Whitney is a separate sensitivity analysis of distributions; it is not automatically a median test. Small samples use the exact method without ties; small tied samples use reproducible permutations (up to 9,999); otherwise an asymptotic test is used. The chosen method is reported. See [SciPy Mann–Whitney methods](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mannwhitneyu.html).

Holm correction spans all four conditions and all selected outcomes, separately for each test family. Unavailable tests remain NaN. At least two subjects per group are required; this is a computational minimum, not a sample-size recommendation. Small cohorts give unstable estimates; assess distributions and study design before inference. Choose outcomes before inspecting p-values. No automatic normality-test-based test selection is used. See [SciPy Welch test and CI](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html).


In [ ]:
between_groups = compare_groups(subjects, metrics=OUTCOMES)
display(between_groups)


## 5. Paired condition changes and group differences in those changes

Within each group, compare blind minus normal separately for dominant and nondominant trials, and nondominant minus dominant separately for normal and blind trials. Only subjects with both measurements enter a contrast. Paired t-tests use one difference per subject and have Holm adjustment across all requested contrasts/outcomes/groups.

The second table compares these **subject-level changes** between disease and control using Welch tests (difference of differences). This explores group-by-condition effects without pretending repeated conditions are independent. Its Holm correction is a separate family spanning all contrasts/outcomes. These contrasts are not a full mixed-effects factorial model and do not adjust for age, medication, task duration, or other covariates.


In [ ]:
paired_effects, change_group_differences = paired_condition_effects(subjects, metrics=OUTCOMES)
display(paired_effects)
display(change_group_differences)


## 6. Replay hand landmarks and inspect individual trials

Select a trial and click **Load replay**. Play/pause and the slider follow recording time; rotate/zoom the view with the mouse. Frames are capped to limit notebook size, and missing detections are blanked rather than interpolated. The skeleton shows 21 landmarks with hand connections; z is estimated wrist-relative depth, so this is not a measured physical 3D scene. Anatomical labels are used when available; legacy detection indices remain explicitly unidentified.

The plots below the replay show the selected task hand's relative trajectory, pinch distance, and speed. If metric configuration is missing, replay still works. All JavaScript is embedded locally; no data or animation requires a network service.

The animation highlights key insertion in gold, labels the event time, marks it on the time slider, and provides a **Key in lock** jump button. Time-series plots show a vertical event line; the trajectory marks the selected fingertip at that time when detections are available. These indicate event timing, not a measured key/object location.

The review controls contain just the insertion time and notes. Success is assumed. Correct the time, click **Save event time**, reload the replay, and rerun analysis cells to update results.


In [ ]:
# Retire the old widget/view before creating a replacement.
if "trial_review_widget" in globals():
    trial_review_widget.close()
trial_review_widget = review_widget(
    inventory, hand_selections=hand_selections,
    allow_legacy=ALLOW_LEGACY_HAND_INDEX, legacy_image_size=LEGACY_IMAGE_SIZE,
    min_confidence=MIN_HANDEDNESS_CONFIDENCE, max_gap_seconds=MAX_DERIVATIVE_GAP_SECONDS,
)
display(trial_review_widget)


## 7. Export tables and analysis configuration

Exports go to `analysis_exports/` beside `data/`; acquisition CSVs are preserved. Re-running replaces these review exports. They contain participant names, so treat them like the source data. No example/synthetic observations are mixed into study results.


In [ ]:
if not inventory.empty:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    tables = {
        "inventory": inventory, "trial_metrics": trial_metrics, "subject_metrics": subjects,
        "descriptives": descriptives, "group_comparisons": between_groups,
        "paired_effects": paired_effects, "group_change_comparisons": change_group_differences,
        "whole_trial_subject_metrics": whole_trial_subjects,
        "whole_trial_descriptives": whole_trial_descriptives,
        "whole_trial_group_comparisons": whole_trial_comparisons,
        "overall_subject_metrics": overall_subjects,
        "overall_descriptives": overall_descriptives,
        "overall_group_comparisons": overall_comparisons,
    }
    for name, table in tables.items():
        table.to_csv(EXPORT_DIR / f"{name}.csv", index=False)
    if not subjects.empty:
        distribution_figure.savefig(EXPORT_DIR / "group_distributions.png", dpi=200, bbox_inches="tight")
        distribution_figure.savefig(EXPORT_DIR / "group_distributions.pdf", bbox_inches="tight")
        paired_figure.savefig(EXPORT_DIR / "paired_conditions.png", dpi=200, bbox_inches="tight")
    configuration = {
        "data_dir": str(DATA_DIR), "outcomes": OUTCOMES, "whole_trial_outcomes": WHOLE_TRIAL_OUTCOMES,
        "dominant_hands": DOMINANT_HANDS, "hand_selections": hand_selections,
        "allow_legacy_hand_index": ALLOW_LEGACY_HAND_INDEX,
        "legacy_image_size": LEGACY_IMAGE_SIZE, "min_frames": MIN_FRAMES,
        "min_coverage": MIN_COVERAGE, "min_duration_seconds": MIN_DURATION_SECONDS,
        "min_handedness_confidence": MIN_HANDEDNESS_CONFIDENCE,
        "max_derivative_gap_seconds": MAX_DERIVATIVE_GAP_SECONDS,
        "successful_tasks_only": SUCCESSFUL_TASKS_ONLY, "min_rest_phase_frames": MIN_REST_PHASE_FRAMES,
    }
    (EXPORT_DIR / "configuration.json").write_text(json.dumps(configuration, indent=2) + "\n")
    print(f"Exported tables, figures, and configuration to {EXPORT_DIR}")
else:
    print("No acquired data to export yet.")
